# Thành viên 2 — Kaggle `GPU T4 x 2` (32 GB VRAM): Track Quốc tế `PHEME` (`Twitter-RoBERTa + mDeBERTa Stance Verbalizer -> SBi-FAGAT`)

**Cấu hình bắt buộc trên Kaggle trước khi chạy song song với Thành viên 1:**
1. Bật **Accelerator: GPU T4 x 2** (tổng 32 GB VRAM).
2. Bật **Internet: ON** để tải `PHEME` từ Figshare (có kiểm tra MD5) và trọng số Hugging Face.

Toàn bộ kết quả thực nghiệm được xuất sạch vào **`results/`** (tuyệt đối không ghi đè hay xóa tệp trong `paper/`).

Bạn có 2 chế độ chạy song song:
- **`member2_en`** *(Khuyên dùng để chia 50/50 với Thành viên 1 — Tier 2A PHEME `K=10, min_total_replies=15`)*: Chạy trọn gói dữ liệu chuẩn quốc tế `PHEME` (`cardiffnlp/twitter-roberta-base-offensive` + `mDeBERTa-v3-xnli` Stance Verbalizer -> chưng cất Cross-Encoder 2D `cardiffnlp/twitter-roberta-base` -> trích xuất `.pt` *Ancestor SpanPool* -> 5-seed công bằng 15+ mô hình + `SBiFAGAT + Tier2Prior` trên Inductive + Leave-One-Event-Out + Ablation + `--include-e2e` Efficiency + Tier 2 PHEME 10 mô hình; kiểm định stance EN qua RumourEval SDQC).
- **`model_gnn_only`**: Chuyên trách train & tinh chỉnh **Mạng Đồ thị `SBi-FAGAT` + 14 mô hình so sánh** (nếu đã nạp `processed/vn` từ Thành viên 1 hoặc bật `--bootstrap-if-missing` để chạy ngay không cần chờ).

In [ ]:
# Bước 1: Kiểm tra GPU NVIDIA Tesla T4
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| Num GPUs:", torch.cuda.device_count())

In [ ]:
%%bash
set -euo pipefail
# Bước 2: Clone/cập nhật repo từ GitHub & cài gói (khóa đúng torch CUDA của Kaggle)
BRANCH="${SBIF_BRANCH:-main}"
cd /kaggle/working
if [ ! -d "gnn-nlp-propagation/.git" ]; then
  rm -rf gnn-nlp-propagation
  git clone https://github.com/hieu-kiien/gnn-nlp-propagation.git
fi
cd /kaggle/working/gnn-nlp-propagation
git fetch origin
git checkout -B "$BRANCH" "origin/$BRANCH"
python -c 'import torch; print("torch==" + torch.__version__.split("+")[0])' > /tmp/kaggle_constraints.txt
grep -v '^torch>=' requirements.txt > /tmp/req_kaggle.txt
pip install -q -c /tmp/kaggle_constraints.txt -r /tmp/req_kaggle.txt
pytest -q --ignore=tests/e2e

In [ ]:
# Khởi tạo muối băm SHA-256 bền vững (nếu chưa đặt SBIF_HASH_SALT) cho giả danh hóa tác giả PHEME
import os, pathlib, secrets
os.chdir("/kaggle/working/gnn-nlp-propagation")
salt_file = pathlib.Path("data/interim/.author_hash_salt.secret")
if not os.environ.get("SBIF_HASH_SALT") and not salt_file.exists():
    salt_file.parent.mkdir(parents=True, exist_ok=True)
    salt_file.write_text(secrets.token_hex(16), encoding="utf-8")
    print("Created persistent salt at", salt_file)

In [ ]:
# Bước 3: Kiểm thử khói toàn bộ đường ống (xong trong <10 giây)
!python -m scripts.kaggle_parallel_runner --role member2_en --smoke-test

In [ ]:
# Bước 4: CHẠY CHÍNH THỨC TRACK 2 PHEME TIER 2A (K=10, N>=15) + RUMOUREVAL SDQC TRÊN KAGGLE
import pathlib, shlex, shutil, subprocess, sys, zipfile
ROLE = "member2_en"  # hoặc "model_gnn_only"
LANG = "en"          # đổi thành "vi" nếu nạp processed/vn từ Thành viên 1
K_HORIZON = 10 if LANG == "en" else 20
MIN_REPLIES = 15 if LANG == "en" else 25
RESULTS_DIR = "results"
CLEAN_PREVIOUS_RESULTS = False
RUN_UPFD = False     # Bật True nếu muốn chạy lại UPFD GossipCop/PolitiFact
INCLUDE_HEAVY_WEIGHTS = True
DISTILL_FROM_FEATURES = False  # Bật True nếu đã mount sẵn data/processed/pheme/nlp_features.pt
USE_HOSTILITY_ROUTER = True
INCLUDE_TIER2_PRIOR = True
RUN_TIER2_MACRO = True
MP_GRAPH_MODE = "coalesced"

res_root = pathlib.Path(RESULTS_DIR)
res_root.mkdir(parents=True, exist_ok=True)
if CLEAN_PREVIOUS_RESULTS and ROLE == "member2_en" and LANG == "en":
    for old_p in sorted(res_root.glob("*_pheme.*")):
        old_p.unlink(missing_ok=True)
    old_fig_dir = res_root / "figures" / "pheme"
    if old_fig_dir.exists():
        shutil.rmtree(old_fig_dir, ignore_errors=True)
    old_fig_dir.mkdir(parents=True, exist_ok=True)

flags = [
    f"--role {ROLE}",
    f"--lang {LANG}",
    f"--k-horizon {K_HORIZON}",
    f"--min-total-replies {MIN_REPLIES}",
    "--hidden-dim 64",
    "--epochs-nlp 3",
    "--epochs-gnn 100",
    f"--results-dir {RESULTS_DIR}",
]
if RUN_UPFD:
    flags.append("--run-upfd")
if INCLUDE_HEAVY_WEIGHTS:
    flags.append("--include-heavy-weights")
if DISTILL_FROM_FEATURES:
    flags.append("--distill-from-features")
if USE_HOSTILITY_ROUTER:
    flags.append("--hostility-target-router")
if INCLUDE_TIER2_PRIOR:
    flags.append("--include-tier2-prior")
if RUN_TIER2_MACRO:
    flags.append(f"--run-tier2 --mp-graph-mode {MP_GRAPH_MODE}")

cmd = "-m scripts.kaggle_parallel_runner " + " ".join(flags)
subprocess.run([sys.executable, "-u", *shlex.split(cmd)], check=True)

# Đánh giá chuẩn ngoại vi SemEval-2019 Task 7 (RumourEval SDQC) theo source claim (--context-mode root) & parent
subprocess.run([
    sys.executable, "-u", "-m", "scripts.eval_rumoureval_sdqc",
    "--split", "test",
    "--context-mode", "root",
    "--output-json", f"{RESULTS_DIR}/results_rumoureval_sdqc.json",
], check=True)
subprocess.run([
    sys.executable, "-u", "-m", "scripts.eval_rumoureval_sdqc",
    "--split", "test",
    "--context-mode", "parent",
    "--output-json", f"{RESULTS_DIR}/results_rumoureval_sdqc_parent.json",
], check=True)
subprocess.run([
    sys.executable, "-u", "-m", "scripts.assemble_paper_artifacts",
    "--paper-dir", RESULTS_DIR,
], check=True)

bundle_zip = pathlib.Path(f"sbifagat_{ROLE}_bundle.zip")
refreshed_files = (
    f"{RESULTS_DIR}/results_rumoureval_sdqc.json",
    f"{RESULTS_DIR}/results_rumoureval_sdqc_parent.json",
    f"{RESULTS_DIR}/tables_human_audit.tex",
    f"{RESULTS_DIR}/tables_dataset_eda.tex",
    f"{RESULTS_DIR}/tables_efficiency.tex",
    f"{RESULTS_DIR}/assembly_manifest.json",
)
if bundle_zip.is_file():
    tmp_zip = bundle_zip.with_suffix(".tmp.zip")
    present_updates = {rel: pathlib.Path(rel) for rel in refreshed_files if pathlib.Path(rel).is_file()}
    with zipfile.ZipFile(bundle_zip, "r") as zin, zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
        for info in zin.infolist():
            if info.filename not in present_updates:
                zout.writestr(info, zin.read(info.filename))
        for rel, fp in present_updates.items():
            zout.write(fp, arcname=rel)
    tmp_zip.replace(bundle_zip)
    if pathlib.Path("/kaggle/working").exists():
        shutil.copy2(bundle_zip, pathlib.Path("/kaggle/working") / bundle_zip.name)
        print("Exported bundle to /kaggle/working/" + bundle_zip.name)
sealed_zip = pathlib.Path(f"sbifagat_{ROLE}_sealed_machine_keys.zip")
if sealed_zip.is_file() and pathlib.Path("/kaggle/working").exists():
    shutil.copy2(sealed_zip, pathlib.Path("/kaggle/working") / sealed_zip.name)

In [ ]:
# Bước 5 (Tùy chọn): Chạy bổ sung UPFD PolitiFact nếu bật RUN_UPFD = True
if globals().get("RUN_UPFD", False):
    import pathlib, shutil, subprocess, sys, zipfile
    out_dir = globals().get("RESULTS_DIR", "results")
    upfd_tex = f"{out_dir}/tables_upfd_politifact.tex"
    upfd_json = f"{out_dir}/results_upfd_politifact.json"
    res = subprocess.run([
        sys.executable, "-m", "scripts.run_upfd_extension",
        "--dataset-name", "politifact",
        "--feature", "bert",
        "--hidden-dim", "64",
        "--output-table", upfd_tex,
        "--output-json", upfd_json,
    ])
    if res.returncode == 0:
        bundle_zip = pathlib.Path(f"sbifagat_{ROLE}_bundle.zip")
        if bundle_zip.is_file():
            upfd_updates = {
                rel: pathlib.Path(rel)
                for rel in (upfd_tex, upfd_json)
                if pathlib.Path(rel).is_file()
            }
            tmp_zip = bundle_zip.with_suffix(".upfd.tmp.zip")
            with zipfile.ZipFile(bundle_zip, "r") as zin, zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
                for info in zin.infolist():
                    if info.filename not in upfd_updates:
                        zout.writestr(info, zin.read(info.filename))
                for rel, fp in upfd_updates.items():
                    zout.write(fp, arcname=rel)
            tmp_zip.replace(bundle_zip)
            if pathlib.Path("/kaggle/working").exists():
                shutil.copy2(bundle_zip, pathlib.Path("/kaggle/working") / bundle_zip.name)
            print("Appended UPFD PolitiFact artifacts into", bundle_zip, "and refreshed /kaggle/working/")
    else:
        print("[WARN] Optional UPFD PolitiFact exited with code", res.returncode, "(primary PHEME bundle preserved).")
else:
    print("Skipping optional UPFD PolitiFact rerun (RUN_UPFD=False).")

## Xuất kết quả gộp với Thành viên 1
Trước khi tắt máy ảo Kaggle, tải **2 gói nén** trong `/kaggle/working/`:
1. `sbifagat_member2_en_bundle.zip` — gồm `results/results_main_pheme.json`, `results/results_loo_pheme.json`, `results/results_ablation_pheme.json`, `results/results_efficiency_pheme.json`, `results/results_macro_network_pheme.json`, `results/results_signed_link_pheme.json`, `results/results_rumoureval_sdqc.json`, `results/results_upfd.json`, `results/tables_*_pheme.tex`, `results/tables_upfd.tex`, `checkpoints/best_sbifagat_pheme.pt`, trọng số NLP (`checkpoints/dual_ce_pheme/dual_head_weights.pt`, `data/processed/pheme/nlp_features.pt` khi bật `--include-heavy-weights`), `results/figures/pheme/*.pdf` và phiếu kiểm định EN (tùy chọn; kiểm định stance tiếng Anh chính thức dùng RumourEval SDQC).
2. `sbifagat_member2_en_sealed_machine_keys.zip` — gói niêm phong riêng chứa `*_machine_key.csv` (giữ riêng để đối chiếu `machine_vs_gold` nếu chạy audit cặp EN).

*Lưu ý:* Ở chế độ mặc định công bằng không chạy `auto_loop` (thêm `--run-auto-loop` ở Bước 4 nếu muốn xuất thêm `outputs/auto_loop_pheme/{loop_diagnostics,worst_val_cases_for_review}.json`).